In [1]:
import MDAnalysis as mda
from monomer_classes import *
from tqdm import tqdm
import yaml

/Users/nate_schwindt/anaconda3/envs/mda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_charge(frag, charges_dict):
    return round(np.sum([charges_dict[frag][atom]['charge'] for atom in charges_dict[frag]]),8)

In [3]:
# Check yaml charges for each fragment type
with open('charges.yaml', 'r') as file: # read in charges from yaml file
        charges_dict = yaml.safe_load(file)

for frag in charges_dict:
    charge = get_charge(frag, charges_dict)
    print(frag, charge)

MPD_Cl_1 -0.0
MPD_Cl_2 -0.0
MPD_2Cl 0.0
MPD_L -0.15680002
MPD_L_Cl_ortho -0.19916003
MPD_T -0.07696003
MPD_T_Cl_1 -0.11239
MPD_T_Cl_2 -0.11239
TMC_L_P_1 0.15392007
TMC_L_P_2 0.15392007
TMC_L_P_3 0.15392007
TMC_L_D_1 -0.74802001
TMC_L_D_2 -0.74802001
TMC_L_D_3 -0.74802001
TMC_T_2P_1 0.07840002
TMC_T_2P_2 0.07840002
TMC_T_2P_3 0.07840002
TMC_T_1P_1_2 -0.87691007
TMC_T_1P_1_3 -0.87691007
TMC_T_1P_2_1 -0.87691007
TMC_T_1P_2_3 -0.87691007
TMC_T_1P_3_1 -0.87691007
TMC_T_1P_3_2 -0.87691007
TMC_T_0P_1 -1.85471996
TMC_T_0P_2 -1.85471996
TMC_T_0P_3 -1.85471996
TMC_C 0.19542
TMC_0P -3.0
TMC_1P_1 -2.0
TMC_1P_2 -2.0
TMC_1P_3 -2.0
TMC_2P_1 -1.0
TMC_2P_2 -1.0
TMC_2P_3 -1.0
TMC_3P 0.0


In [4]:
u = mda.Universe('chlorinated.tpr', 'chlorinated.gro')
atoms = u.select_atoms('resname PA*')
print(f'Charge before reassignment: {atoms.charges.sum()}')

Charge before reassignment: -136.08628137334017


In [5]:
# get the number of each fragment
print('Creating MPD monomer fragments...')
mpds = []
Ns = []
for N in tqdm(atoms.select_atoms('type n nh')):

    has_cl = 'cl' in N.bonded_atoms.types

    if not has_cl: # check other N
        my_ca = [atom for atom in N.bonded_atoms if atom.type == 'ca'][0] # get connected aromatic C
        next_ca = [atom for atom in my_ca.bonded_atoms if atom.type == 'ca'] # get next aromatic C
        for ca in next_ca:
            next_next_ca = [atom for atom in ca.bonded_atoms if atom != my_ca and atom.type == 'ca'][0]
            if 'N' in next_next_ca.bonded_atoms.elements:
                n = [atom for atom in next_next_ca.bonded_atoms if atom.element == 'N'][0]
                break

        has_cl = 'cl' in n.bonded_atoms.types

    if has_cl:
        mpd = MPD_Cl_T(N, cl='cl', xlink_n='n', ar_c='ca', term_n='nh', ar_h='ha', hn='hn')
    else:
        mpd = MPD(N, xlink_n='n', ar_c='ca', term_n='nh', ar_h='ha', hn='hn')

    if mpd.N2 not in Ns:
        mpd.assign_charges(charges_dict[mpd.name])
        mpds.append(mpd)
    
    Ns.append(mpd.N1)
    Ns.append(mpd.N2)

print('Creating TMC monomer fragments...')
tmcs = []
Cs = []
for C in tqdm(atoms.select_atoms('type c')):
    tmc = TMC(C, ar_c='ca', xlink_c='c', ar_h='ha', deprot_o='o', prot_o='oh', ho_type='ho')
    if tmc.C2 not in Cs and tmc.C3 not in Cs:
        tmc.assign_charges(charges_dict[tmc.name])
        tmcs.append(tmc)

    Cs.append(tmc.C1)
    Cs.append(tmc.C2)
    Cs.append(tmc.C3)


n_mpd_mono = len([mono for mono in mpds if mono.name == 'MPD'])
n_mpd_mono_Cl = len([mono for mono in mpds if mono.name in ['MPD_Cl_1', 'MPD_Cl_2']])
n_mpd_mono_2Cl = len([mono for mono in mpds if mono.name == 'MPD_2Cl'])
n_mpd_L = len([mono for mono in mpds if mono.name == 'MPD_L'])
n_mpd_T = len([mono for mono in mpds if mono.name == 'MPD_T'])
n_mpd_T_Cl = len([mono for mono in mpds if mono.name in ['MPD_T_Cl_1', 'MPD_T_Cl_2']])
n_tmc_mono = len([mono for mono in tmcs if mono.name == 'TMC'])
n_tmc_mono_0P = len([mono for mono in tmcs if mono.name == 'TMC_0P'])
n_tmc_mono_1P = len([mono for mono in tmcs if mono.name in ['TMC_1P_1', 'TMC_1P_2', 'TMC_1P_3']])
n_tmc_mono_2P = len([mono for mono in tmcs if mono.name in ['TMC_2P_1', 'TMC_2P_2', 'TMC_2P_3']])
n_tmc_mono_3P = len([mono for mono in tmcs if mono.name == 'TMC_3P'])
n_tmc_LD = len([mono for mono in tmcs if mono.name in ['TMC_L_D_1', 'TMC_L_D_2', 'TMC_L_D_3']])
n_tmc_LP = len([mono for mono in tmcs if mono.name in ['TMC_L_P_1', 'TMC_L_P_2', 'TMC_L_P_3']])
n_tmc_T0P = len([mono for mono in tmcs if mono.name in ['TMC_T_0P_1', 'TMC_T_0P_2', 'TMC_T_0P_3']])
n_tmc_T1P = len([mono for mono in tmcs if mono.name in ['TMC_T_1P_1_2', 'TMC_T_1P_1_3', 'TMC_T_1P_2_1', 'TMC_T_1P_2_3', 'TMC_T_1P_3_1', 'TMC_T_1P_3_2']])
n_tmc_T2P = len([mono for mono in tmcs if mono.name in ['TMC_T_2P_1', 'TMC_T_2P_2', 'TMC_T_2P_3']])
n_tmc_C = len([mono for mono in tmcs if mono.name == 'TMC_C'])

mpd_L = get_charge('MPD_L', charges_dict)
mpd_T = get_charge('MPD_T', charges_dict)
mpd_T_Cl = get_charge('MPD_T_Cl_1', charges_dict)
tmc_LD = get_charge('TMC_L_D_1', charges_dict)
tmc_LP = get_charge('TMC_L_P_1', charges_dict)
tmc_T0P = get_charge('TMC_T_0P_1', charges_dict)
tmc_T1P = get_charge('TMC_T_1P_1_2', charges_dict)
tmc_T2P = get_charge('TMC_T_2P_1', charges_dict)
tmc_C = get_charge('TMC_C', charges_dict)

print(f'\nMPD monomers: {n_mpd_mono + n_mpd_mono_Cl + n_mpd_mono_2Cl}')
print(f'MPD monomer: {n_mpd_mono} with charge: {n_mpd_mono*0:.4f}')
print(f'MPD monomer-Cl: {n_mpd_mono_Cl} with charge: {n_mpd_mono_Cl*0:.4f}')
print(f'MPD monomer-2Cl: {n_mpd_mono_2Cl} with charge: {n_mpd_mono_2Cl*0:.4f}')
print(f'MPD-L: {n_mpd_L} with charge: {n_mpd_L*mpd_L:.4f}')
print(f'MPD-T: {n_mpd_T} with charge: {n_mpd_T*mpd_T:.4f}')
print(f'MPD-T-Cl: {n_mpd_T_Cl} with charge: {n_mpd_T_Cl*mpd_T_Cl:.4f}')
print(f'\nTMC monomers: {n_tmc_mono + n_tmc_mono_0P + n_tmc_mono_1P + n_tmc_mono_2P + n_tmc_mono_3P}')
print(f'TMC monomer (0P): {n_tmc_mono_0P} with charge: {n_tmc_mono_0P*(-3):.4f}')
print(f'TMC monomer (1P): {n_tmc_mono_1P} with charge: {n_tmc_mono_1P*(-2):.4f}')
print(f'TMC monomer (2P): {n_tmc_mono_2P} with charge: {n_tmc_mono_2P*(-1):.4f}')
print(f'TMC monomer (3P): {n_tmc_mono_3P} with charge: {n_tmc_mono_3P*(0):.4f}')
print(f'TMC-L-D: {n_tmc_LD} with charge: {n_tmc_LD*tmc_LD:.4f}')
print(f'TMC-L-P: {n_tmc_LP} with charge: {n_tmc_LP*tmc_LP:.4f}')
print(f'TMC-T-0P: {n_tmc_T0P} with charge: {n_tmc_T0P*tmc_T0P:.4f}')
print(f'TMC-T-1P: {n_tmc_T1P} with charge: {n_tmc_T1P*tmc_T1P:.4f}')
print(f'TMC-T-2P: {n_tmc_T2P} with charge: {n_tmc_T2P*tmc_T2P:.4f}')
print(f'TMC-C: {n_tmc_C} with charge: {n_tmc_C*tmc_C:.4f}')

n_deprot = n_tmc_LD + n_tmc_T0P*2 + n_tmc_T1P
print(f'\nTotal number of deprotonated groups on polymer chains: {n_deprot}')
print(f'Charge after reassignment: {atoms.charges.sum()}')


Creating MPD monomer fragments...


100%|██████████| 1432/1432 [00:10<00:00, 132.54it/s]


Creating TMC monomer fragments...


100%|██████████| 1464/1464 [00:08<00:00, 179.85it/s]


MPD monomers: 23
MPD monomer: 0 with charge: 0.0000
MPD monomer-Cl: 11 with charge: 0.0000
MPD monomer-2Cl: 12 with charge: 0.0000
MPD-L: 452 with charge: -70.8736
MPD-T: 87 with charge: -6.6955
MPD-T-Cl: 154 with charge: -17.3081

TMC monomers: 6
TMC monomer (0P): 2 with charge: -6.0000
TMC monomer (1P): 0 with charge: 0.0000
TMC monomer (2P): 3 with charge: -3.0000
TMC monomer (3P): 1 with charge: 0.0000
TMC-L-D: 92 with charge: -68.8178
TMC-L-P: 107 with charge: 16.4694
TMC-T-0P: 22 with charge: -40.8038
TMC-T-1P: 15 with charge: -13.1537
TMC-T-2P: 14 with charge: 1.0976
TMC-C: 232 with charge: 45.3374

Total number of deprotonated groups on polymer chains: 151
Charge after reassignment: -163.74803497000008


In [6]:
# Set up system of equations to solve for new charges
# Note: this is solving for the total charge on each fragment, not per-atom charges
#
# Let the sum of the charges on a given fragment be represented as follows:
# MPD_L = m0, MPD_T = m1, MPD_T_Cl = m2, TMC_L_P = t1, TMC_L_D = t2, 
# TMC_T_0P = t3, TMC_T_1P = t4, TMC_T_2P = t5, TMC_C = t6
# 
# m0 = \sum{charges on MPD_L}, which we constrain to be exactly as in OpenEye
# m0 = 2*t5
# m1 = m2 + 0.03606 # this difference comes from the OpenEye charges for chlorinated and unchlorinated MPD-T
# 2*m1 = t1
# 3*m1 = t6
# t2 = t1 - 1
# t3 = t5 - 2
# t4 = t5 -1
# Total charge = -n_deprot 
# 
# This gives a system of 8 equations and 8 unknowns, which we represent as a matrix equation Ax = b
# where x = [m1, m2, t1, t2, t3, t4, t5, t6]

A = np.array([[0,  0,  0, 0, 0, 0,  2,  0],
              [1, -1,  0, 0, 0, 0,  0,  0],
              [2,  0, -1, 0, 0, 0,  0,  0],
              [3,  0,  0, 0, 0, 0,  0, -1],
              [0,  0, -1, 1, 0, 0,  0,  0],
              [0,  0,  0, 0, 1, 0, -1,  0],
              [0,  0,  0, 0, 0, 1, -1,  0],
              [n_mpd_T, n_mpd_T_Cl, n_tmc_LP, n_tmc_LD, n_tmc_T0P, n_tmc_T1P, n_tmc_T2P, n_tmc_C]])

b = np.array([[mpd_L],
              [0.03606],
              [0],
              [0],
              [-1],
              [-2],
              [-1],
              [-n_mpd_L*mpd_L - n_deprot]])

print('Solving system of equations for new fragment charges...')
x = np.linalg.solve(A, b)

Solving system of equations for new fragment charges...


In [7]:
# distribute the differences in charge evenly over the 6 aromatic carbons in each fragment
# dump to a new charges dict: charges_modified.yaml
diff = {}

frags = ['MPD_T', 'MPD_T_Cl', 'TMC_L_P', 'TMC_L_D', 'TMC_T_0P', 'TMC_T_1P', 'TMC_T_2P', 'TMC_C']
orig_charges = [mpd_T, mpd_T_Cl, tmc_LP, tmc_LD, tmc_T0P, tmc_T1P, tmc_T2P, tmc_C]
for frag, charge, orig in zip(frags,x,orig_charges):
    d = (charge[0]-orig) / 6
    for name in charges_dict:
        if name.startswith(frag):
            diff[name] = d
    
    print(f'{frag} should have {charge[0]:.5f} charge for a difference of {d*6:.5f} from the original')
    print(f'\tDivided evenly over 6 aromatic carbons: {d:.5f}')

for frag in charges_dict:
    if frag in diff:
        for atom in charges_dict[frag]:
            if atom.startswith('ca'):
                charges_dict[frag][atom]['charge'] = float(round(charges_dict[frag][atom]['charge'],8) + diff[frag])

with open('charges_modified.yaml', 'w') as f:
    yaml.dump(charges_dict, f)

MPD_T should have 0.06024 charge for a difference of 0.13720 from the original
	Divided evenly over 6 aromatic carbons: 0.02287
MPD_T_Cl should have 0.02418 charge for a difference of 0.13657 from the original
	Divided evenly over 6 aromatic carbons: 0.02276
TMC_L_P should have 0.12049 charge for a difference of -0.03343 from the original
	Divided evenly over 6 aromatic carbons: -0.00557
TMC_L_D should have -0.87951 charge for a difference of -0.13149 from the original
	Divided evenly over 6 aromatic carbons: -0.02192
TMC_T_0P should have -2.07840 charge for a difference of -0.22368 from the original
	Divided evenly over 6 aromatic carbons: -0.03728
TMC_T_1P should have -1.07840 charge for a difference of -0.20149 from the original
	Divided evenly over 6 aromatic carbons: -0.03358
TMC_T_2P should have -0.07840 charge for a difference of -0.15680 from the original
	Divided evenly over 6 aromatic carbons: -0.02613
TMC_C should have 0.18073 charge for a difference of -0.01469 from the ori

In [8]:
# get the number of each fragment
print('Creating MPD monomer fragments...')
mpds = []
Ns = []
for N in tqdm(atoms.select_atoms('type n nh')):

    has_cl = 'cl' in N.bonded_atoms.types

    if not has_cl: # check other N
        my_ca = [atom for atom in N.bonded_atoms if atom.type == 'ca'][0] # get connected aromatic C
        next_ca = [atom for atom in my_ca.bonded_atoms if atom.type == 'ca'] # get next aromatic C
        for ca in next_ca:
            next_next_ca = [atom for atom in ca.bonded_atoms if atom != my_ca and atom.type == 'ca'][0]
            if 'N' in next_next_ca.bonded_atoms.elements:
                n = [atom for atom in next_next_ca.bonded_atoms if atom.element == 'N'][0]
                break

        has_cl = 'cl' in n.bonded_atoms.types

    if has_cl:
        mpd = MPD_Cl_T(N, cl='cl', xlink_n='n', ar_c='ca', term_n='nh', ar_h='ha', hn='hn')
    else:
        mpd = MPD(N, xlink_n='n', ar_c='ca', term_n='nh', ar_h='ha', hn='hn')

    if mpd.N2 not in Ns:
        mpd.assign_charges(charges_dict[mpd.name])
        mpds.append(mpd)
    
    Ns.append(mpd.N1)
    Ns.append(mpd.N2)

print('Creating TMC monomer fragments...')
tmcs = []
Cs = []
for C in tqdm(atoms.select_atoms('type c')):
    tmc = TMC(C, ar_c='ca', xlink_c='c', ar_h='ha', deprot_o='o', prot_o='oh', ho_type='ho')
    if tmc.C2 not in Cs and tmc.C3 not in Cs:
        tmc.assign_charges(charges_dict[tmc.name])
        tmcs.append(tmc)

    Cs.append(tmc.C1)
    Cs.append(tmc.C2)
    Cs.append(tmc.C3)


n_mpd_mono = len([mono for mono in mpds if mono.name == 'MPD'])
n_mpd_mono_Cl = len([mono for mono in mpds if mono.name in ['MPD_Cl_1', 'MPD_Cl_2']])
n_mpd_mono_2Cl = len([mono for mono in mpds if mono.name == 'MPD_2Cl'])
n_mpd_L = len([mono for mono in mpds if mono.name == 'MPD_L'])
n_mpd_T = len([mono for mono in mpds if mono.name == 'MPD_T'])
n_mpd_T_Cl = len([mono for mono in mpds if mono.name in ['MPD_T_Cl_1', 'MPD_T_Cl_2']])
n_tmc_mono = len([mono for mono in tmcs if mono.name == 'TMC'])
n_tmc_mono_0P = len([mono for mono in tmcs if mono.name == 'TMC_0P'])
n_tmc_mono_1P = len([mono for mono in tmcs if mono.name in ['TMC_1P_1', 'TMC_1P_2', 'TMC_1P_3']])
n_tmc_mono_2P = len([mono for mono in tmcs if mono.name in ['TMC_2P_1', 'TMC_2P_2', 'TMC_2P_3']])
n_tmc_mono_3P = len([mono for mono in tmcs if mono.name == 'TMC_3P'])
n_tmc_LD = len([mono for mono in tmcs if mono.name in ['TMC_L_D_1', 'TMC_L_D_2', 'TMC_L_D_3']])
n_tmc_LP = len([mono for mono in tmcs if mono.name in ['TMC_L_P_1', 'TMC_L_P_2', 'TMC_L_P_3']])
n_tmc_T0P = len([mono for mono in tmcs if mono.name in ['TMC_T_0P_1', 'TMC_T_0P_2', 'TMC_T_0P_3']])
n_tmc_T1P = len([mono for mono in tmcs if mono.name in ['TMC_T_1P_1_2', 'TMC_T_1P_1_3', 'TMC_T_1P_2_1', 'TMC_T_1P_2_3', 'TMC_T_1P_3_1', 'TMC_T_1P_3_2']])
n_tmc_T2P = len([mono for mono in tmcs if mono.name in ['TMC_T_2P_1', 'TMC_T_2P_2', 'TMC_T_2P_3']])
n_tmc_C = len([mono for mono in tmcs if mono.name == 'TMC_C'])

mpd_L = get_charge('MPD_L', charges_dict)
mpd_T = get_charge('MPD_T', charges_dict)
mpd_T_Cl = get_charge('MPD_T_Cl_1', charges_dict)
tmc_LD = get_charge('TMC_L_D_1', charges_dict)
tmc_LP = get_charge('TMC_L_P_1', charges_dict)
tmc_T0P = get_charge('TMC_T_0P_1', charges_dict)
tmc_T1P = get_charge('TMC_T_1P_1_2', charges_dict)
tmc_T2P = get_charge('TMC_T_2P_1', charges_dict)
tmc_C = get_charge('TMC_C', charges_dict)

print(f'\nMPD monomers: {n_mpd_mono + n_mpd_mono_Cl + n_mpd_mono_2Cl}')
print(f'MPD monomer: {n_mpd_mono} with charge: {n_mpd_mono*0:.4f}')
print(f'MPD monomer-Cl: {n_mpd_mono_Cl} with charge: {n_mpd_mono_Cl*0:.4f}')
print(f'MPD monomer-2Cl: {n_mpd_mono_2Cl} with charge: {n_mpd_mono_2Cl*0:.4f}')
print(f'MPD-L: {n_mpd_L} with charge: {n_mpd_L*mpd_L:.4f}')
print(f'MPD-T: {n_mpd_T} with charge: {n_mpd_T*mpd_T:.4f}')
print(f'MPD-T-Cl: {n_mpd_T_Cl} with charge: {n_mpd_T_Cl*mpd_T_Cl:.4f}')
print(f'\nTMC monomers: {n_tmc_mono + n_tmc_mono_0P + n_tmc_mono_1P + n_tmc_mono_2P + n_tmc_mono_3P}')
print(f'TMC monomer (0P): {n_tmc_mono_0P} with charge: {n_tmc_mono_0P*(-3):.4f}')
print(f'TMC monomer (1P): {n_tmc_mono_1P} with charge: {n_tmc_mono_1P*(-2):.4f}')
print(f'TMC monomer (2P): {n_tmc_mono_2P} with charge: {n_tmc_mono_2P*(-1):.4f}')
print(f'TMC monomer (3P): {n_tmc_mono_3P} with charge: {n_tmc_mono_3P*(0):.4f}')
print(f'TMC-L-D: {n_tmc_LD} with charge: {n_tmc_LD*tmc_LD:.4f}')
print(f'TMC-L-P: {n_tmc_LP} with charge: {n_tmc_LP*tmc_LP:.4f}')
print(f'TMC-T-0P: {n_tmc_T0P} with charge: {n_tmc_T0P*tmc_T0P:.4f}')
print(f'TMC-T-1P: {n_tmc_T1P} with charge: {n_tmc_T1P*tmc_T1P:.4f}')
print(f'TMC-T-2P: {n_tmc_T2P} with charge: {n_tmc_T2P*tmc_T2P:.4f}')
print(f'TMC-C: {n_tmc_C} with charge: {n_tmc_C*tmc_C:.4f}')

n_deprot = n_tmc_LD + n_tmc_T0P*2 + n_tmc_T1P
print(f'\nTotal number of deprotonated groups on polymer chains: {n_deprot}')
print(f'Charge after reassignment: {atoms.charges.sum()}')


Creating MPD monomer fragments...


100%|██████████| 1432/1432 [00:10<00:00, 139.40it/s]


Creating TMC monomer fragments...


100%|██████████| 1464/1464 [00:08<00:00, 164.34it/s]


MPD monomers: 23
MPD monomer: 0 with charge: 0.0000
MPD monomer-Cl: 11 with charge: 0.0000
MPD monomer-2Cl: 12 with charge: 0.0000
MPD-L: 452 with charge: -70.8736
MPD-T: 87 with charge: 5.2412
MPD-T-Cl: 154 with charge: 3.7243

TMC monomers: 6
TMC monomer (0P): 2 with charge: -6.0000
TMC monomer (1P): 0 with charge: 0.0000
TMC monomer (2P): 3 with charge: -3.0000
TMC monomer (3P): 1 with charge: 0.0000
TMC-L-D: 92 with charge: -80.9152
TMC-L-P: 107 with charge: 12.8921
TMC-T-0P: 22 with charge: -45.7248
TMC-T-1P: 15 with charge: -16.1760
TMC-T-2P: 14 with charge: -1.0976
TMC-C: 232 with charge: 41.9296

Total number of deprotonated groups on polymer chains: 151
Charge after reassignment: -159.99999103000005


In [9]:
# Check yaml charges for each fragment type
with open('charges_modified.yaml', 'r') as file: # read in charges from yaml file
        charges_dict = yaml.safe_load(file)

for frag in charges_dict:
    charge = get_charge(frag, charges_dict)
    print(frag, charge)

MPD_2Cl -0.0
MPD_Cl_1 -0.0
MPD_Cl_2 0.0
MPD_L -0.15680002
MPD_L_Cl_ortho -0.19916003
MPD_T 0.06024363
MPD_T_Cl_1 0.02418363
MPD_T_Cl_2 0.02418363
TMC_0P -3.0
TMC_1P_1 -2.0
TMC_1P_2 -2.0
TMC_1P_3 -2.0
TMC_2P_1 -1.0
TMC_2P_2 -1.0
TMC_2P_3 -1.0
TMC_3P 0.0
TMC_C 0.1807309
TMC_L_D_1 -0.87951273
TMC_L_D_2 -0.87951273
TMC_L_D_3 -0.87951273
TMC_L_P_1 0.12048727
TMC_L_P_2 0.12048727
TMC_L_P_3 0.12048727
TMC_T_0P_1 -2.07840001
TMC_T_0P_2 -2.07840001
TMC_T_0P_3 -2.07840001
TMC_T_1P_1_2 -1.07840001
TMC_T_1P_1_3 -1.07840001
TMC_T_1P_2_1 -1.07840001
TMC_T_1P_2_3 -1.07840001
TMC_T_1P_3_1 -1.07840001
TMC_T_1P_3_2 -1.07840001
TMC_T_2P_1 -0.07840001
TMC_T_2P_2 -0.07840001
TMC_T_2P_3 -0.07840001
